Cell 1: Install Required Libraries

In [1]:
!pip install -q transformers torch

Cell 2: Load the Tokenizer and Model

In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load a small, fast open model suitable for CPU execution
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set model to evaluation mode (disables dropout layers)
model.eval()
print(f"Loaded Model: {model_name} | Vocabulary size: {tokenizer.vocab_size:,} tokens")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded distilgpt2 | Vocabulary size: 50,257 tokens


Cell 3: Run Forward Pass & Inspect Top-K Next-Token Probabilities

In [7]:
def inspect_next_tokens(prompt: str, top_k: int = 10):
    # 1. Convert text into token IDs (PyTorch tensors)
    inputs = tokenizer(prompt, return_tensors="pt")

    # 2. Forward pass without tracking gradients (saves memory & speeds up inference)
    with torch.no_grad():
        outputs = model(**inputs)

    # 3. Extract logits for the VERY LAST token position
    # outputs.logits shape: [batch_size, sequence_length, vocab_size]
    next_token_logits = outputs.logits[0, -1, :]

    # 4. Convert raw logits into probabilities (0.0 to 1.0) using Softmax
    probabilities = F.softmax(next_token_logits, dim=-1)

    # 5. Get the top-k highest probabilities and their corresponding token IDs
    top_probs, top_indices = torch.topk(probabilities, top_k)

    # 6. Display results in a clean table
    print(f"Prompt: '{prompt}'\n")
    print(f"{'Rank':<6} {'Token':<18} {'Token ID':<10} {'Raw Logit':<12} {'Probability':<12}")
    print("-" * 62)

    for rank, (prob, idx) in enumerate(zip(top_probs, top_indices), start=1):
        # repr() makes leading spaces (like ' Paris') visible
        token_str = repr(tokenizer.decode(idx))
        raw_logit = next_token_logits[idx].item()
        print(f"{rank:<6} {token_str:<18} {idx.item():<10} {raw_logit:<12.4f} {prob.item():>6.2f}")

# Test the function with an incomplete sentence
inspect_next_tokens("The capital of France is", top_k=10)

Prompt: 'The capital of France is'

Rank   Token              Token ID   Raw Logit    Probability 
--------------------------------------------------------------
1      ' the'             262        -57.3821       0.12
2      ' home'            1363       -57.7406       0.08
3      ' now'             783        -58.2412       0.05
4      ' a'               257        -58.2520       0.05
5      ' being'           852        -58.5779       0.04
6      ' one'             530        -59.1009       0.02
7      ' known'           1900       -59.2094       0.02
8      ' also'            635        -59.2149       0.02
9      ' in'              287        -59.2623       0.02
10     ' to'              284        -59.5085       0.01


Classroom Exercises to Try in Cell 4

In [6]:
# Exercise 1: High confidence / factual completion
inspect_next_tokens("Water boils at 100 degrees", top_k=5)

print("\n" + "="*62 + "\n")

# Exercise 2: High uncertainty / open-ended creative prompt
inspect_next_tokens("Once upon a time, there was a", top_k=5)

Prompt: 'Water boils at 100 degrees'

Rank   Token              Token ID   Raw Logit    Probability 
--------------------------------------------------------------
1      ' Fahrenheit'      35935      -58.9696       0.22%
2      ','                11         -59.3260       0.15%
3      ' Celsius'         34186      -59.4020       0.14%
4      '.'                13         -59.7986       0.09%
5      ' and'             290        -60.1684       0.07%


Prompt: 'Once upon a time, there was a'

Rank   Token              Token ID   Raw Logit    Probability 
--------------------------------------------------------------
1      ' lot'             1256       -60.0627       0.03%
2      ' time'            640        -60.1259       0.03%
3      ' great'           1049       -60.3577       0.02%
4      ' new'             649        -60.4455       0.02%
5      ' certain'         1728       -60.5176       0.02%
